In [6]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/pizza_rl_pj

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/pizza_rl_pj


In [7]:
from pathlib import Path

if __name__ == "__main__":
    output = Path("/content/drive/MyDrive/pizza_rl_pj/results")
    output.mkdir(parents=True, exist_ok=True)

    q, history = train()

In [8]:
from pathlib import Path
import csv
import numpy as np
from pizza_env import PizzaKitchenEnv


def greedy_action(q_values, rng):
    # Random tie-breaking prevents a systematic bias toward action 0 (Idle).
    return int(rng.choice(np.flatnonzero(q_values == q_values.max())))


def train(episodes=30000, seed=42, alpha=0.1, gamma=0.99,
          epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.9997):
    env = PizzaKitchenEnv(observation_mode="discrete")
    rng = np.random.default_rng(seed)
    q = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float32)
    history = []
    for episode in range(episodes):
        state, _ = env.reset(seed=seed + episode)
        done = False
        epsilon = max(epsilon_end, epsilon_start * epsilon_decay ** episode)
        while not done:
            if rng.random() < epsilon:
                action = int(rng.integers(env.action_space.n))
            else:
                action = greedy_action(q[state], rng)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            target = reward if done else reward + gamma * np.max(q[next_state])
            q[state, action] += alpha * (target - q[state, action])
            state = next_state
        history.append({"episode": episode + 1, "reward": info["episode_return"],
                        "served": info["served"], "expired": info["expired"],
                        "wrong_serves": info["wrong_serves"]})
    env.close()
    return q, history


def evaluate(q, episodes=500, seed=100000):
    env = PizzaKitchenEnv(observation_mode="discrete")
    rng = np.random.default_rng(seed)
    results = []
    for episode in range(episodes):
        state, _ = env.reset(seed=seed + episode)
        done = False
        while not done:
            action = greedy_action(q[state], rng)
            state, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        results.append(info)
    env.close()
    return {
        "avg_reward": float(np.mean([r["episode_return"] for r in results])),
        "order_completion_rate": float(sum(r["served"] for r in results) / (episodes * 2)),
        "avg_expired_orders": float(np.mean([r["expired"] for r in results])),
        "avg_wrong_serves": float(np.mean([r["wrong_serves"] for r in results])),
        "avg_steps": float(np.mean([r["steps"] for r in results])),
    }


# Google Drive의 결과 저장 폴더
output = Path("/content/drive/MyDrive/pizza_rl_pj/results")
output.mkdir(parents=True, exist_ok=True)

# Q-learning 학습
q, history = train()

# Q-table 저장
np.save(output / "q_table.npy", q)

# 학습 기록 CSV 저장
with (output / "q_learning_history.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)

# 학습된 Q-table 평가
print("Evaluation:", evaluate(q))
print("Saved:", output)


Evaluation: {'avg_reward': 16.185399999999994, 'order_completion_rate': 0.614, 'avg_expired_orders': 0.772, 'avg_wrong_serves': 0.0, 'avg_steps': 6.306}
Saved: /content/drive/MyDrive/pizza_rl_pj/results


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
from pizza_env import PizzaKitchenEnv

# 1. 파일 경로 및 환경 설정
output_dir = Path("/content/drive/MyDrive/pizza_rl_pj/results")
q_table = np.load(output_dir / "q_table.npy")
env = PizzaKitchenEnv(observation_mode="discrete")

# 메뉴 이름 매핑
MENU_NAMES = {0: "None", 1: "Cheese Pizza (1,2)", 2: "Pepperoni Pizza (1,2,3)"}
ACTION_NAMES = ["0: Idle", "1: Dough (도우)", "2: Cheese (치즈)", "3: Pepperoni (페퍼로니)", "4: Serve (서빙)"]

def greedy_action(q_values):
    return int(np.argmax(q_values))

# 2. Step-by-Step 추적 실행
def run_step_by_step_trace(seed=1004):
    obs, info = env.reset(seed=seed)
    done = False
    logs = []

    print("=" * 70)
    print(f"  [Q-Learning 에이전트 플레이 Step-by-Step 시뮬레이션] (Seed: {seed})")
    print("=" * 70)

    while not done:
        raw_state = info["raw_state"]
        plate = list(raw_state[:3])
        o1_menu, o1_time = raw_state[3], raw_state[4]
        o2_menu, o2_time = raw_state[5], raw_state[6]

        # 최적 행동 선택
        action = greedy_action(q_table[obs])

        # 환경 1 스텝 진행
        next_obs, reward, terminated, truncated, next_info = env.step(action)
        done = terminated or truncated

        # 로그 기록
        step_num = info["steps"] + 1
        event_name = next_info["event"]

        logs.append({
            "Step": step_num,
            "조리대 (Plate)": str(plate),
            "주문 1 (Menu, Time)": f"M{o1_menu} (T={o1_time})" if o1_menu != 0 else "완료",
            "주문 2 (Menu, Time)": f"M{o2_menu} (T={o2_time})" if o2_menu != 0 else "완료",
            "선택 행동 (Action)": ACTION_NAMES[action],
            "보상 (Reward)": f"{reward:+.1f}",
            "이벤트 (Event)": event_name
        })

        obs = next_obs
        info = next_info

    df_trace = pd.DataFrame(logs)
    return df_trace, info

# 3. 시뮬레이션 실행 및 표 출력
df_result, final_info = run_step_by_step_trace(seed=1004)

# 콘솔 출력 (DataFrame)
print(df_result.to_string(index=False))

print("-" * 70)
print(f" 최종 결과 | 총 Reward: {final_info['episode_return']:.1f} | "
      f"제출 성공(Served): {final_info['served']} | 만료(Expired): {final_info['expired']}")
print("-" * 70)

# Markdown 표
print("\n [Markdown 표]\n")
print(df_result.to_markdown(index=False))

  [Q-Learning 에이전트 플레이 Step-by-Step 시뮬레이션] (Seed: 1004)
 Step 조리대 (Plate) 주문 1 (Menu, Time) 주문 2 (Menu, Time) 선택 행동 (Action) 보상 (Reward) 이벤트 (Event)
    1   [0, 0, 0]          M2 (T=4)          M1 (T=4)  1: Dough (도우)        -0.1       dough
    2   [1, 0, 0]          M2 (T=3)          M1 (T=3) 2: Cheese (치즈)        -0.1      cheese
    3   [1, 1, 0]          M2 (T=2)          M1 (T=2)  4: Serve (서빙)       +19.9      served
    4   [0, 0, 0]          M2 (T=1)                완료        0: Idle       -10.1        idle
----------------------------------------------------------------------
 최종 결과 | 총 Reward: 9.6 | 제출 성공(Served): 1 | 만료(Expired): 1
----------------------------------------------------------------------

 [Markdown 표]

|   Step | 조리대 (Plate)   | 주문 1 (Menu, Time)   | 주문 2 (Menu, Time)   | 선택 행동 (Action)   |   보상 (Reward) | 이벤트 (Event)   |
|-------:|:-----------------|:----------------------|:----------------------|:---------------------|----------------:|:-----------------|
| 